# 03 — Detecting the duplicate: previous-token, duplicate-token & induction heads

Notebook 02 mapped the **back** of the IOI circuit: the **name movers** (9.6, 9.9, 10.0) that attend from END to the indirect object and copy it, and the **S-inhibition heads** (7.3, 7.9, 8.6, 8.10) that steer those movers *away* from the subject. But S-inhibition can only inhibit the subject if something upstream has already worked out **which name is the repeated one**. Given

> "When **John** and Mary went to the store, **John** gave a drink to ___"

what tells the model that `John` (at S2, position 10) is a duplicate of `John` (at S1, position 2)? That's the **front** of the circuit, and it's what this notebook finds.

The model has **two parallel mechanisms** for noticing the repeat — redundancy is a recurring theme in this circuit:

| family | heads (Wang et al.) | what it does |
|---|---|---|
| **duplicate-token** | 0.1, 0.10, 3.0 | at S2, attend straight back to S1 — "I've seen this exact token" |
| **previous-token** | 2.2, 4.11 | copy "the token at i−1" into position i — *plumbing* for induction |
| **induction** | 5.5, 5.8, 5.9, 6.9 | use the previous-token signal to find the earlier copy via `[A][B]…[A]→[B]` |

Both the duplicate-token and induction families write a "this position holds a repeated token" signal into **S2's residual stream**, which the S-inhibition heads then read. Chain: **detect duplicate (here) → inhibit subject (02) → move the remaining name (02).**

We replicate the head identities from [Wang et al. (2022)](https://arxiv.org/abs/2211.00593) — find them yourself first, then check against the paper.

In [ ]:
import sys
sys.path.insert(0, "..")          # repo root, so `interp_lab` imports

import torch
from interp_lab import load_model, ioi_task, logit_diff   # the new shared module

model = load_model()              # GPT-2 small, eval, grad off
task = ioi_task(model)            # clean/corrupted tokens + positions + answer toks
NL, NH = model.cfg.n_layers, model.cfg.n_heads
print(f"layers={NL} heads={NH}")
print(f"positions: S1={task.S1_POS} S2={task.S2_POS} IO={task.IO_POS} END={task.END}")
print(list(enumerate(task.str_tokens)))

## Setup — now from the shared module

First notebook to import `interp_lab` instead of re-pasting the loader, prompts,
positions, and metric. (00–02 keep their inline copies — they were written to
stand alone.) The named positions, for reference:

- **`S1_POS` = 2** — first `" John"`.
- **`S2_POS` = 10** — second `" John"` (the duplicate we're hunting the detector for).
- **`IO_POS` = 4** — `" Mary"` (the answer; the name movers' target).
- **`END` = 14** — final `" to"`.

Cache the clean run; pin the metric. Everything below reads attention patterns
off this one cache.

In [ ]:
clean_logits, clean_cache = model.run_with_cache(task.clean_tokens)
CLEAN_LD = logit_diff(clean_logits, task)
print(f"clean logit_diff: {CLEAN_LD:+.3f}   (expect ≈ +3.17 — IOI behavior present)")
print("attn pattern shape [head, q, k]:", tuple(clean_cache["pattern", 0][0].shape))

## Exercise 1 — duplicate-token heads (S2 → S1)

The most direct way to flag the repeat: a head sitting at **S2** that attends
**back to S1** — the earlier token that is literally identical. `cache["pattern", L]`
is `[batch, head, query, key]`; we want `query = S2_POS`, `key = S1_POS`, swept
over every head.

**Predict first.** Of 144 heads, how many do you expect to put real attention on
the S2→S1 link? Which *layers* — early or late? (Detecting a duplicate is a
precondition for everything downstream, so it has to happen early.)

In [ ]:
dup = torch.zeros(NL, NH)
# TODO: dup[L, H] = attention weight from S2_POS (query) to S1_POS (key),
#       for head H of layer L, on the clean run.
...

from interp_lab import heatmap
heatmap(dup, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "duplicate-token: attention S2 → S1")

<details><summary><b>Solution — Exercise 1</b></summary>

```python
for L in range(NL):
    dup[L] = clean_cache["pattern", L][0, :, task.S2_POS, task.S1_POS]
```

`pattern[0, :, S2, S1]` is one slice: batch 0, all heads, query fixed at S2, key
fixed at S1.

**Checkpoint.** Three heads light up in the early layers — **0.1, 0.10, 3.0** —
each putting a large share of S2's attention straight onto S1. Everything else is
near zero. These are the **duplicate-token heads**: position-based "have I seen
this token already?" detectors. (Exact weights vary a little by setup; the *set*
is the result.)
</details>

### Why early, and why position-based

A duplicate-token head's QK circuit matches on **token identity + a "this appeared
before" signal**, not on meaning. It fires in layer 0–3 because the inhibition and
mover machinery downstream needs the answer in hand by the middle layers. It's the
cheapest of the two detectors — no plumbing required, just a key matching a key
it already saw.

## Exercise 2 — previous-token heads (i → i−1)

The second detector is indirect and needs a feeder. **Previous-token heads** don't
find the duplicate themselves; they copy *"the token one position back"* into each
position, laying down the rails that induction heads ride. Signature: at every
query position `i`, attention concentrated on `key = i−1` — the **sub-diagonal**
of the pattern.

**Predict.** A previous-token head's attention pattern is a single bright line just
below the diagonal. Score each head by the mean of that sub-diagonal; which heads
top it?

In [ ]:
prevtok = torch.zeros(NL, NH)
# TODO: prevtok[L, H] = mean attention from position i to position i-1, averaged
#       over query positions, for each head. (Hint: torch diagonal at offset=-1.)
...

heatmap(prevtok, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "previous-token: mean attention i → i-1")

<details><summary><b>Solution — Exercise 2</b></summary>

```python
for L in range(NL):
    p = clean_cache["pattern", L][0]               # [head, q, k]
    prevtok[L] = p.diagonal(dim1=-2, dim2=-1, offset=-1).mean(-1)
```

`diagonal(offset=-1)` picks `pattern[h, i, i-1]` for every `i`; the mean over
those is the "how much does this head look one token back" score.

**Checkpoint.** **2.2** and **4.11** dominate — the previous-token heads. On their
own they do nothing for IOI; their output is the input the induction heads read in
the next exercise.
</details>

## Exercise 3 — induction heads (isolate them off the IOI prompt)

Induction is the `[A][B] … [A] → [B]` mechanism: having seen `A` followed by `B`
once, when `A` recurs the head attends to **the token that came after the first
`A`**. It's cleanest to isolate on a **repeated random sequence** rather than the
IOI prompt — no semantics, pure structure.

Build `[BOS] r₀ r₁ … r_{n-1} r₀ r₁ … r_{n-1}`. On the second copy, an induction
head at position of the repeated `rⱼ` should attend to **`rⱼ₊₁`'s first
occurrence** — a stripe parallel to the diagonal, offset by `-(n-1)`.

**Predict.** Which heads, and in which layers? (Induction is built *on top of*
previous-token heads, so it must live a layer or two later than 2.2 / 4.11.)

In [ ]:
seq_len = 50
rand = torch.randint(1000, 10000, (1, seq_len), device=model.cfg.device)
bos = torch.tensor([[model.tokenizer.bos_token_id]], device=model.cfg.device)
rep_tokens = torch.cat([bos, rand, rand], dim=1)        # [1, 2*seq_len + 1]
_, rep_cache = model.run_with_cache(rep_tokens)

induction = torch.zeros(NL, NH)
# TODO: induction[L, H] = mean of the induction stripe — attention at
#       offset=-(seq_len-1) — for each head, on rep_cache.
...

heatmap(induction, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "induction score (repeated-sequence test)")

<details><summary><b>Solution — Exercise 3</b></summary>

```python
for L in range(NL):
    p = rep_cache["pattern", L][0]                                  # [head, q, k]
    stripe = p.diagonal(dim1=-2, dim2=-1, offset=-(seq_len - 1))    # [head, q']
    induction[L] = stripe.mean(-1)
```

The stripe at `offset=-(seq_len-1)` is exactly "attend back to one-after-the-first-
copy" — the induction signature.

**Checkpoint.** **5.5, 5.8, 5.9, 6.9** top the grid — the induction heads, sitting
right above the previous-token heads that feed them. Re-run the duplicate read
(Ex 1) for these heads on the IOI prompt and you'll see they *also* attend S2→S1
there: induction is the **second**, parallel route to "John is a repeat."
</details>

### Two detectors, one job

Duplicate-token (0.1/0.10/3.0) and induction (5.5/5.8/5.9/6.9) reach the same
conclusion by different means — direct token-match vs. previous-token-then-match.
That redundancy is why ablating *one* family barely dents the behavior, which is
exactly the next test.

## Exercise 4 — the causal link: knock out the front, watch the back

Correlation so far: these heads *attend* to the duplicate. The causal claim: they
**write the signal that S-inhibition reads off S2**. Test it — ablate the
duplicate **and** induction heads together (zero their `z`), then measure the IOI
metric. If they feed the inhibition→mover chain, removing them should collapse it.

**Predict.** Ablate *only* duplicate-token heads: big effect or small? (Recall the
redundancy.) Now ablate duplicate **+** induction together — what happens to the
logit diff?

In [ ]:
from collections import defaultdict

DUP   = [(0, 1), (0, 10), (3, 0)]
INDUC = [(5, 5), (5, 8), (5, 9), (6, 9)]

def ablate(head_list):
    by_layer = defaultdict(list)
    for L, H in head_list:
        by_layer[L].append(H)
    def mk(L):
        def hook(act, hook):            # act: [batch, pos, head, d_head]
            for H in by_layer[L]:
                act[:, :, H, :] = 0.0
            return act
        return hook
    fwd = [(f"blocks.{L}.attn.hook_z", mk(L)) for L in by_layer]
    return logit_diff(model.run_with_hooks(task.clean_tokens, fwd_hooks=fwd), task)

# TODO: print CLEAN_LD, ablate(DUP), ablate(INDUC), ablate(DUP+INDUC)
...

<details><summary><b>Solution — Exercise 4</b></summary>

```python
print(f"clean              {CLEAN_LD:+.3f}")
print(f"− duplicate only   {ablate(DUP):+.3f}")
print(f"− induction only   {ablate(INDUC):+.3f}")
print(f"− both families    {ablate(DUP + INDUC):+.3f}")
```

**Reading.** Either family alone barely moves the logit diff — each is a backup for
the other (same lesson as the backup name movers in 02). Ablating **both** is what
visibly degrades it: with no "S2 is the duplicate" signal written, S-inhibition has
nothing to read, the movers are no longer steered off the subject, and the gap
shrinks. The front of the circuit is load-bearing only when you remove *both*
parallel paths — redundancy hides single-point importance, which is why ablation
(not just patching-in) is the honest test.
</details>

## Exercise 5 — break it on purpose

Two falsification tests. Predict each, then run.

**(a) Remove the duplicate.** Use a prompt whose subject is a *fresh* name:
`"When John and Mary went to the store, Sue gave a drink to"`. Now nothing is
duplicated. The duplicate-token / induction heads should go quiet at the (former)
S2 slot — there's no repeat to detect.

**(b) Widen the gap.** Pad the sentence so S1 and S2 are far apart. Do the
detectors still link them? (Duplicate-token heads match on identity regardless of
distance; induction depends on the previous-token rails surviving the gap.)

In [ ]:
no_dup = "When John and Mary went to the store, Sue gave a drink to"
nd_tokens = model.to_tokens(no_dup)
_, nd_cache = model.run_with_cache(nd_tokens)

# TODO (a): read the duplicate-token heads' attention from S2_POS to S1_POS on
#           nd_cache and compare to the clean run — expect it to collapse.
# TODO (b): construct a longer prompt with the same two names far apart and
#           re-check 0.1 / 3.0 / induction heads.
...

<details><summary><b>Solution — Exercise 5</b></summary>

```python
for L, H in DUP:
    clean_w = clean_cache["pattern", L][0, H, task.S2_POS, task.S1_POS].item()
    nd_w    = nd_cache["pattern", L][0, H, task.S2_POS, task.S1_POS].item()
    print(f"{L}.{H}:  clean {clean_w:.2f}  →  no-dup {nd_w:.2f}")
```

**Reading.** With a non-repeated subject the duplicate-token attention at that slot
collapses — confirming these heads fire *on the repeat*, not on the position or
the name. (b) Duplicate-token heads still find the match across a wide gap — they
key on identity. Induction is the more fragile of the two: it rides the
previous-token rails, so a disrupted local structure hurts it more. The redundancy
is the point: two mechanisms with different failure modes.
</details>

## The circuit, end to end

With the front mapped, the whole IOI story closes:

```
previous-token (2.2, 4.11)
        │  lay down "token at i−1" rails
        ▼
induction (5.5, 5.8, 5.9, 6.9)  ──┐
                                  ├──►  write "S2 is a duplicate" into S2's residual
duplicate-token (0.1, 0.10, 3.0) ──┘
        │
        ▼
S-inhibition (7.3, 7.9, 8.6, 8.10)   read S2, write "don't copy the subject" into END
        │
        ▼
name movers (9.6, 9.9, 10.0)         attend END → IO, copy " Mary"
        (with negative movers 10.7, 11.10 and a bench of backups)
```

Detect the duplicate → inhibit the subject → move the remaining name. Every arrow
is a measured attention pattern or a causal ablation, not a just-so story — that's
the whole point of doing it by hand.

This is also the right moment to read [Wang et al. (2022)](https://arxiv.org/abs/2211.00593)
front to back: every head it names now corresponds to a grid cell you produced
yourself.

**Next — `04`:** a different question entirely. Not "what circuit computes IOI" but
"what features does the residual stream encode in a human-readable basis." Enter
sparse autoencoders.